In [ ]:
import scanpy as sc
import squidpy as sq
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import anndata as ad
from scipy import sparse
import anndata as ad
import json

In [ ]:
%cd './src'

import importlib
import ssgsea
import ssgsea_plots
import multimodal
import clustering

importlib.reload(ssgsea)
importlib.reload(ssgsea_plots)
importlib.reload(multimodal)
importlib.reload(clustering)
%cd ..

In [ ]:
# load the preprocessed data
adata = ad.read_h5ad("BRCA_preprocessed.h5ad", backed=None)  
library_id='Visium_Human_Breast_Cancer'
adata

In [ ]:
# run the image segmentation for all values of sigma, save in a new anndata object
adata_img = multimodal.run_img_segmentation(
                adata = adata,
                output_dir = "segmentation_results",
                sigmas = [0, 0.5, 1, 2, 4, 6, 8, 10],
                superpixel = "slic",
                res = "hires"
                )

In [ ]:
# load the anndata object with the image segmentation results
adata_img = ad.read_h5ad("segmentation_results/adata_img_segmentation_slic.h5ad")

In [ ]:
# run the multimodal pipeline using gene expression with correlation distance and complete linkage

multimodal.multimodal_distance(    
    adata_img_segmentation = adata_img,
    adata_gene_expr = adata,
    distance_gexpr='correlation',
    use_pca=True,
    modality_combinations = [True, False], # how I want to combine gexpr and gsets
    superpixel = 'slic',
    sigmas = [0, 0.5, 1, 2, 4, 6, 8, 10],
    weight_combinations = [(1,1), (0.25, 0.75), (0.75, 0.25)],
    cluster = True,
    linkage = 'complete',
    k = [8, 16, 20, 32],
    output_dir = "multimodal_results_gexpr_correlation"
    )

In [ ]:
# plot the multimodal clustering results

results_adata_path = Path('multimodal_results_gexpr_correlation/')
dirs = list(results_adata_path.glob('*'))
for d in dirs:
    inp = str(d)
    out = str(d)+'/plots'
    clustering.show_results(inp, out)